<a href="https://colab.research.google.com/github/SunJ2432/ai-image-compliance/blob/main/notebooks/03_descriptor_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A3 — Image description + subclassification
Qwen3.5-4B (vision) and Qwen2.5-1.5B-Instruct (classification). Run on a Colab T4. Models are loaded **sequentially** to reduce VRAM pressure.

In [1]:
%pip install -q -U "transformers>=4.57" accelerate safetensors pillow pandas pytest timm

If dependencies changed, restart the Colab runtime before continuing. Do not install `torchao` for this workflow.

In [16]:
from pathlib import Path
import sys
GITHUB_REPO = "SunJ2432/ai-image-compliance"
repo_dir = Path("/content") / GITHUB_REPO.split("/")[-1]
if not repo_dir.exists():
    !git clone https://github.com/{GITHUB_REPO}.git {repo_dir}
else:
    !git -C {repo_dir} pull --ff-only
assert (repo_dir / "samples_tiny/manifest.csv").is_file(), "Dataset missing; check GitHub sync"
if str(repo_dir / "src") not in sys.path:
    sys.path.insert(0, str(repo_dir / "src"))
print("Repo:", repo_dir)

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.84 KiB | 941.00 KiB/s, done.
From https://github.com/SunJ2432/ai-image-compliance
   2c7413c..7781fee  main       -> origin/main
Updating 2c7413c..7781fee
Fast-forward
 tests/test_understand.py | 70 ++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 70 insertions(+)
 create mode 100644 tests/test_understand.py
Repo: /content/ai-image-compliance


In [3]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Select T4 GPU under Runtime > Change runtime type"

GPU: Tesla T4


## Stage 1 — Caption all images. The Qwen3.5-4B model downloads on first call. The identity-safety instruction is in `captioner/model.py`.

In [4]:
import pandas as pd
from imgcompliance.captioner import Captioner
manifest = repo_dir / "samples_tiny/manifest.csv"
df = pd.read_csv(manifest)
captioner = Captioner()
caption_rows = []
for _, row in df.iterrows():
    path = manifest.parent / row["filename"]
    assert path.exists(), f"Missing image: {path}"
    caption = captioner.describe(path)
    caption_rows.append({"filename": row["filename"], "ground_truth": row["label"], "caption": caption})
    print(path.name, caption["scene_summary"][:120])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


real_001.jpg The user wants a JSON description of the image.
I need to follow specific constraints:
- Only visible evidence.
- No ide
ai_001.jpg The user wants a JSON description of the image.
I need to follow specific constraints:
- Describe based *only* on visibl
real_002.jpg The user wants a JSON description of the image.
I need to follow specific constraints:
- Only visible evidence.
- No ide
ai_002.jpg The user wants a JSON description of the image.
I need to follow specific constraints:
- Describe based *only* on visibl
real_003.jpg The user wants a JSON description of the image.
I need to follow specific constraints:
- Describe based *only* on visibl
ai_003.jpg The user wants a JSON description of the image.
I need to follow strict constraints:
- Only visible evidence.
- No names


## Release the VLM before loading the classifier

In [10]:
print("captioner exists:", "captioner" in globals())
print("caption_rows exists:", "caption_rows" in globals())
print("caption_rows count:", len(caption_rows) if "caption_rows" in globals() else 0)

captioner exists: False
caption_rows exists: True
caption_rows count: 6


In [12]:
import gc
import torch

if "captioner" in globals():
    del captioner

gc.collect()
torch.cuda.empty_cache()

print("Captioning finished; GPU cache cleared")

Captioning finished; GPU cache cleared


## Stage 2 — Restore your A2 checkpoint and classify. If checkpoint is missing after a runtime reset, restore it or rerun A2 training first. The default context does **not** assert that any depicted entity has been verified.

In [13]:
%pip install -U qwen-vl-utils

In [14]:
from imgcompliance.detector import Detector
from imgcompliance.classifier import Classifier
checkpoint = Path("/content/detector_head.pt")
assert checkpoint.exists(), "Restore your A2 checkpoint to /content/detector_head.pt"
detector = Detector(checkpoint=checkpoint)
classifier = Classifier()
results = []
for item in caption_rows:
    path = manifest.parent / item["filename"]
    detection = detector.predict(path)
    context = {"declared_purpose": None, "verified_real_entity": False}
    sub = classifier.classify(item["caption"], detection, context)
    results.append({
        "filename": item["filename"], "ground_truth": item["ground_truth"],
        "detected": detection["label"], "ai_score": round(detection["prob"], 4),
        "scene_summary": item["caption"]["scene_summary"],
        "depicts_people": item["caption"]["depicts_people"],
        "appears_photorealistic": item["caption"]["appears_photorealistic"],
        "possible_real_world_reference": item["caption"]["possible_real_world_reference"],
        "caption_format": item["caption"]["format"],
        "subclass": sub["subclass"], "rationale": sub["rationale"]
    })
results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("/content/a3_results.csv", index=False)
print("Saved /content/a3_results.csv")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

,filename,ground_truth,detected,ai_score,scene_summary,depicts_people,appears_photorealistic,possible_real_world_reference,caption_format,subclass,rationale
0,real/real_001.jpg,real,real,0.4396,The user wants a JSON description of the image...,None,None,None,plain_text_fallback,not_ai_generated,Detector classified the image as real; not ind...
1,ai_generated/ai_001.jpg,ai_edited,ai,0.5855,The user wants a JSON description of the image...,None,None,None,plain_text_fallback,general_synthetic,"Based on the provided evidence, the image appe..."
2,real/real_002.jpg,real,real,0.4880,The user wants a JSON description of the image...,None,None,None,plain_text_fallback,not_ai_generated,Detector classified the image as real; not ind...
3,ai_generated/ai_002.jpg,ai_edited,ai,0.6495,The user wants a JSON description of the image...,None,None,None,plain_text_fallback,general_synthetic,"Based on the visual analysis provided, the ima..."
4,real/real_003.jpg,real,real,0.3546,The user wants a JSON description of the image...,None,None,None,plain_text_fallback,not_ai_generated,Detector classified the image as real; not ind...
5,ai_generated/ai_003.jpg,ai_edited,real,0.4683,The user wants a JSON description of the image...,None,None,None,plain_text_fallback,not_ai_generated,Detector classified the image as real; not ind...


Saved /content/a3_results.csv


## Unit tests (no model downloads)

In [17]:
!PYTHONPATH={repo_dir}/src python -m pytest -q {repo_dir}/tests/test_understand.py

.............                                                            [100%]
13 passed in 0.03s


**Interpretation:** This is a smoke test using the same six images as A2 training. It does not establish real-world accuracy. A deepfake subclass needs external evidence of a real-world entity; the landscape samples should generally be `general_synthetic` when the detector says `ai`.